# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# Cell 1: Imports and setup
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [47]:
# Cell 2: Initialize Groq client
load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key and groq_api_key.startswith('gsk_') and len(groq_api_key) > 10:
    print("Groq API key looks good so far")
else:
    print("There might be a problem with your Groq API key!")

MODEL = 'openai/gpt-oss-20b'
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key
)

Groq API key looks good so far


In [24]:

links = fetch_website_links("https://edwarddonner.com")
print(f"Found {len(links)} links")
links[:10] 

Found 31 links


['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral']

In [25]:
# Cell 4: Link selection system prompt
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [26]:
# Cell 5: Function to create user prompt for link selection
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [27]:
# Cell 6: Test the user prompt
print(get_links_user_prompt("https://edwarddonner.com")[:500] + "...")


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/ab...


In [28]:

def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    
    try:
        response = groq_client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": link_system_prompt},
                {"role": "user", "content": get_links_user_prompt(url)}
            ],
            response_format={"type": "json_object"}
        )
        
        result = response.choices[0].message.content
        links = json.loads(result)
        
        print(f"Found {len(links.get('links', []))} relevant links")
        
        # Show token usage
        if hasattr(response, 'usage'):
            print(f"Tokens used: {response.usage.total_tokens}")
            
        return links
    except Exception as e:
        print(f"Error: {e}")
        return {"links": []}

In [29]:

select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling openai/gpt-oss-120b
Found 10 relevant links
Tokens used: 1573


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'proficient page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [30]:
# Cell 9: Test on another website
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b


Found 6 relevant links
Tokens used: 1331


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'}]}

In [31]:
# Cell 10: Function to fetch page and all relevant links
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    
    result = f"## Landing Page:\n\n{contents}\n\n## Relevant Links:\n"
    
    for link in relevant_links.get('links', []):
        result += f"\n\n### Link: {link['type']}\n"
        try:
            link_content = fetch_website_contents(link["url"])
            result += link_content[:2000] + "..."  # Truncate long pages
        except:
            result += f"[Could not fetch {link['url']}]\n"
    
    return result

In [32]:
# Cell 11: Test fetching all content
content = fetch_page_and_all_relevant_links("https://huggingface.co")


Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 8 relevant links
Tokens used: 1722


In [33]:
print(content)

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
Storage Buckets: AI-native object storage
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Lightricks/LTX-2.3
Updated
6 days ago
•
345k
•
482
Qwen/Qwen3.5-9B
Updated
9 days ago
•
1.39M
•
709
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
3 days ago
•
30.8k
•
389
HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive
Updated
7 days ago
•
127k
•
311
sarvamai/sarvam-105b
Updated
about 23 hours ago
•
4.2k
•
213
Browse 2M+ models
Spaces
Running
on
Zero
Featured
487
Omni Video Factory
🏆
487
text to video, image to video, video extend
Running
on
Zero
183
OBLITERATUS
💥
183
One-click model l

In [ ]:
# Cell 12: Brochure system prompt
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Humorous version (uncomment to use)
brochure_system_prompt = """
You are an genz , sigma , skibidi rizzler , gyat assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [35]:
# Cell 13: Function to create brochure user prompt
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5000]  # Truncate if more than 5,000 characters
    return user_prompt

In [36]:
# Cell 14: Test brochure user prompt
print(get_brochure_user_prompt("HuggingFace", "https://huggingface.co")[:500] + "...")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 7 relevant links
Tokens used: 1354


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.



You are looking at a company called: HuggingFace
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
Storage Buckets: AI-native object storage
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
The...


In [37]:
# Cell 15: Function to create brochure (non-streaming)
def create_brochure(company_name, url):
    print(f"Creating brochure for {company_name}...")
    
    try:
        response = groq_client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": brochure_system_prompt},
                {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
            ],
        )
        
        result = response.choices[0].message.content
        
        # Show token usage
        if hasattr(response, 'usage'):
            print(f"\nTokens used - Prompt: {response.usage.prompt_tokens}, Completion: {response.usage.completion_tokens}, Total: {response.usage.total_tokens}")
        
        display(Markdown(result))
    except Exception as e:
        print(f"Error creating brochure: {e}")

In [22]:
# Cell 16: Test brochure creation
create_brochure("HuggingFace", "https://huggingface.co")

Creating brochure for HuggingFace...
Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-20b
Found 6 relevant links
Tokens used: 1851


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.



Tokens used - Prompt: 1694, Completion: 1581, Total: 3275


# 🎉 Hugging Face: The AI Hangout That Actually Gives a F*ck

> **Where the world’s smartest *humans* and *machines* meet for a cup of *code* and a side of *giggles***  

## 1️⃣ Meet the Squad

| What we’re all about | Why it matters |
|----------------------|----------------|
| **A 2 M+ model playground** – From Lightricks to Qwen3.5 – it’s like a buffet, but for neural nets. | Build, test, and deploy the next AI breakthrough in seconds. |
| **500k+ datasets** – Open data so you don’t have to start from scratch. | Fuel your model’s appetite without the data‑hungry binge. |
| **1 M+ apps (Spaces)** – Zero‑code, zero‑hassle demos that run on “Zero” (CPU) or “MCP” (GPU). | Show off ideas faster than a TikTok trend. |
| **AI‑native object storage (Buckets)** – Because even AI needs a locker. | Store and serve weights, logs, or the world’s best meme dataset. |
| **Courses & Learning** – From LLM to Robotics to Deep RL, all under one roof. | Become the next ML wizard without the Hogwarts tuition. |
| **New tools** – GGML & llama.cpp now live, HuggingChat Omni, Synthetic Data Playbook, Omni Video Factory. | Keep your toolbox fresh; your future’s brighter. |

---

## 2️⃣ Culture: “Work Hard, Hug Harder”

- **Open‑source first** – We’re not shy. We give everything away, so you can remix, fork, or just brag about it at parties.
- **Community‑driven** – Thousands of collaborators, Discord channels, and a “Hug” for every pull request.  
- **Zero‑Trust, All‑Trust** – We trust your code, but we’ll audit it if it’s going to be *public*.
- **No “water cooler” (but we do have a “data cooler”)** – Coffee‑free, but we’ll keep your GPU hot.

---

## 3️⃣ Customers: The Avengers of AI

| Who’s on the list | What they get |
|-------------------|--------------|
| **Start‑ups** | MVPs in minutes. No need to hire a full dev squad. |
| **Enterprises** | Enterprise‑grade pricing + private model hosting. |
| **Researchers** | Unlimited public models + datasets + a community that will cite your work. |
| **Developers** | One‑click model deployment, SDKs, and the joy of seeing your code run on millions of GPUs. |

If you’re building chatbots, image‑to‑video pipelines, or a next‑gen robot, we’re your playground.

---

## 4️⃣ Investors: Plug Into the Future

- **Rapid adoption** – 2 M+ models, 1 M+ apps = an ecosystem that’s *already* living.  
- **Scalable infrastructure** – Buckets = AI‑native object storage; no vendor lock‑in.  
- **Community‑first revenue streams** – Enterprise, hosting, and API access.  
- **Disruptive moat** – Open‑source + proprietary tooling = hard‑to‑replicate synergy.

Basically, if you want a piece of the AI‑future pie, we’re the dough.

---

## 5️⃣ Recruits: Ready to “Hug” Your Next Career?

> **We’re hiring** – Check the Careers page, but you’ll find roles in:
> - Model Engineering
> - Data Infrastructure
> - Community Management
> - AI Ethics & Safety

We’re looking for people who:
- Speak fluent *Python* and *Git*.
- Love the idea of building the next AI super‑app.
- Can explain a transformer to a 5‑year‑old (or a board of investors).
- Are comfortable being the **face** of the AI community.

> **Apply now** – Your first day will involve a coffee‑free, data‑full onboarding.

---

## 6️⃣ Call‑to‑Action

| Want to: | Do it here |
|----------|------------|
| **Explore 2 M+ models** | [huggingface.co/models](https://huggingface.co/models) |
| **Build a space** | [huggingface.co/spaces](https://huggingface.co/spaces) |
| **Run data on buckets** | [huggingface.co/buckets](https://huggingface.co/buckets) |
| **Learn in 3‑minute courses** | [huggingface.co/learn](https://huggingface.co/learn) |
| **Join the community** | [Discord](https://discord.gg/huggingface) |
| **Invest** | Email: invest@huggingface.co |
| **Apply** | [Careers](https://huggingface.co/careers) |

> **Hugging Face – because building AI should feel like a hug, not a headache.**  

Feel free to drop a *hug* in the comments or send a meme – we’re all about the *feel*!

In [38]:
# Cell 16: Test brochure creation oss-120b
create_brochure("HuggingFace", "https://huggingface.co")

Creating brochure for HuggingFace...
Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 8 relevant links
Tokens used: 1578

Tokens used - Prompt: 1716, Completion: 1428, Total: 3144


# 🤗 Hugging Face – The AI Hangout That’s Got Everyone Talking (and Coding)

### **Your One‑Stop Shop for Models, Datasets, Spaces & Buckets**  
*Think of it as the coolest coworking space for machine‑learning nerds, but with way more emojis and way fewer coffee spills.*

---  

## 🎯 What We Do (in 3‑second elevator pitch)

| **Feature** | **What It Means for You** |
|-------------|---------------------------|
| **Models**  | 2 M+ open‑source gems. From tiny chat‑bots to 27 B‑parameter beasts—pick, fine‑tune, and unleash. |
| **Datasets**| 500 k+ curated collections. Want synthetic data? We’ve got a “Trillions of Tokens” playbook. |
| **Spaces**  | Deploy interactive AI apps with a click. Video‑to‑video, image‑edit, chat playgrounds—no infra headache. |
| **Buckets** | AI‑native object storage that talks the same language as your models (GGML, llama.cpp, you name it). |
| **Community**| Over a million active creators sharing, reviewing, and rating like it’s a TikTok for AI. |

> “The platform where the machine learning community collaborates on models, datasets, and applications.” – *Official tagline, but also our group chat motto.*

---  

## 🤩 Why Everyone’s Obsessed  

- **🚀 Trending Models** – Qwen3.5‑9B with 1.39 M downloads, Lightricks/LTX‑2.3 climbing fast, and the ever‑mysterious “sarvam‑105b” dropping new updates every 23 hrs.  
- **👾 Spaces that Spark** – “Omni Video Factory” (text‑to‑video wizardry), “OBLITERATUS” (one‑click model liberation), and “FireRed Image Edit” (speedy transformer‑powered edits).  
- **🧩 Plug‑and‑Play** – No‑code UI, zero‑GPU “Zero” runtimes, and CPU‑upgrade options for the “I’m on a budget” crowd.  
- **💸 Enterprise‑Ready** – Private repos, SSO, compliance, and custom pricing plans for the big‑biz squad.  

---  

## 🌍 Who’s Hanging Out Here?  

| **Who** | **What They’re Doing** |
|---------|------------------------|
| **ML Researchers** | Publishing state‑of‑the‑art papers and letting the world download their models. |
| **Startup Founders** | Building AI‑first products in days, not months, using Spaces as a rapid‑prototyping sandbox. |
| **Enterprise Teams** | Securing proprietary models behind private Buckets, while still tapping the open‑source ecosystem. |
| **Data‑Hungry Creators** | Mining 500 k+ datasets for everything from synthetic voice to reinforcement‑learning playgrounds. |
| **Students & Hobbyists** | Learning by remixing, forking, and hugging (the face, not the code). |

> **Fun fact:** If you count every “⭐️” on the Hub, you could probably orbit the Earth twice. 🌍✨

---  

## 🎉 Culture: Where “Hug” Isn’t Just a Logo  

- **Open‑Source First** – All code lives publicly; PRs are treated like birthday presents.  
- **Ethical AI DNA** – “Build an open and ethical AI future together” isn’t just a tagline; it’s a weekly stand‑up mantra.  
- **Slack‑Level Transparency** – Company updates are shared on the same forum where community members post memes.  
- **Sigma Vibes** – Engineers are encouraged to work at “their own optimal velocity” (aka “move fast, stay curious”).  
- **Skibidi‑Rizz Mode** – Office (or Zoom) parties include AI‑generated beats, meme‑contests, and occasional “🤖 vs 🤖” hackathons.  

---  

## 📈 For Investors: The Numbers That Make Us *Rizz*  

- **2 M+ models** and **500 k+ datasets** → Network effects keep spiraling.  
- **Enterprise pipeline** growing at double‑digit YoY (private Buckets & SSO deals).  
- **Community‑driven innovation** cuts R&D spend—your money fuels the community, not a secret lab.  
- **Brand Assets**: Signature 🤗 logo, bold #FFD21E and #FF9D00 colors—instant recognizability in the AI arena.  

---  

## 🚀 Careers: Join the Hug Squad  

| **Role** | **What You’ll Do** |
|----------|--------------------|
| **Machine‑Learning Engineer** | Build next‑gen Transformers, improve inference on “Zero” runtimes. |
| **Developer Advocate** | Talk to the community, write tutorials, turn user feedback into product gold. |
| **Product Designer** | Make Spaces look as slick as a TikTok UI while staying ultra‑functional. |
| **Ethics & Policy Lead** | Draft guidelines, ensure models stay fair, inclusive, and safe. |
| **Data Engineer** | Keep Buckets humming, design pipelines for petabytes of synthetic data. |

*Perks:* Unlimited model downloads, remote‑first policy, “AI‑Pet” stipend (yes, you can adopt a virtual pet that learns from you), and a yearly “Hack‑the‑Face” retreat in the Swiss Alps.  

---  

## 🎉 Bottom Line  

Hugging Face isn’t just a platform—it’s a **movement**. If you want to:

- **Build** cutting‑edge AI without reinventing the wheel,  
- **Collaborate** with a global community that actually *answers* your PRs,  
- **Scale** from a hobby project to an enterprise‑grade product,  

then grab a virtual hug 🤗 and join the party.  

**Visit:** https://huggingface.co  
**Follow the hype:** #HuggingFace #AICommunity #OpenSourceRizz  

*Because the future of AI is better when we all hug it together.*  

In [41]:
# Cell 17: Streaming version
def stream_brochure(company_name, url):
    print(f"Creating streaming brochure for {company_name}...")
    
    try:
        stream = groq_client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {"role": "system", "content": brochure_system_prompt},
                {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
            ],
            stream=True
        )
        
        response = ""
        display_handle = display(Markdown(""), display_id=True)
        
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                response += chunk.choices[0].delta.content
                update_display(Markdown(response), display_id=display_handle.display_id)
        
        print("\n✅ Brochure complete!")
        
    except Exception as e:
        print(f"Error streaming brochure: {e}")

In [44]:
# Cell 18: Test streaming version
stream_brochure("HuggingFace", "https://huggingface.co")

Creating streaming brochure for HuggingFace...
Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01khn05qcwfw5v61ahwgfnbmee` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199646, Requested 808. Please try again in 3m16.128s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


# Hugging Face  
**Where AI gets a hug and the world gets a little smarter**

> **“The AI community building the future.”**  
> — The Hugging Face team

---

## Why Hugging Face Rocks (And Why You’ll Love It)

| Feature | What it does | Why it matters |
|---------|--------------|----------------|
| **2 M+ Models** | From transformers to diffusion, the library is a pantry of pure AI goodness | Need a chat bot, a vision model, or a brand‑new LLM? Pick, tweak, deploy—no hassle. |
| **1 M+ Apps** | “Spaces” let you run any model in a browser with zero infrastructure | Test, share, and remix—fast as your Wi‑Fi. |
| **500 k+ Datasets** | Public data for training, evaluation, and experimentation | Build better models, faster. |
| **AI‑Native Storage Buckets** | Scale your data the way you scale your models | No more juggling S3 and your code. |
| **GGML & llama.cpp** | Lightning‑fast inference on anything—CPU, GPU, even your phone | Deploy in the wild without breaking a sweat. |
| **HuggingChat Omni** | Chat with your favorite LLMs, from the browser to your terminal | Bring conversational AI to any project. |
| **Synthetic Data Playbook** | Generate trillions of tokens on a bookshelf you can swipe | Train privacy‑preserving models the easy way. |
| **Enterprise Pricing & Docs** | For the big teams that need compliance, security, and support | Enterprise‑grade tooling for real‑world applications. |

---

## Culture: The Hugging Face Way

- **Open‑Source First** – Every repo on GitHub is a playground. Pull requests are our currency.  
- **Community‑Driven** – 100 k+ contributors, 500 k+ discussions, 10 M+ stars. If it’s not in the chat, it’s probably in the docs.  
- **“Code + Coffee”** – Late‑night merges, pair‑programming, and the occasional meme‑filled Slack thread.  
- **Zero‑Trust, Zero‑Barrier** – Everyone, from PhDs to hobbyists, can drop a model and watch it be adopted worldwide.  
- **“Build. Share. Re‑Build.”** – The cycle is as short as a coffee break.

---

## Who Uses Hugging Face?

| Who | What they do | Why they pick Hugging Face |
|-----|--------------|---------------------------|
| **Enterprise AI teams** | Deploy LLMs in production | Enterprise support, fine‑tuning, secure hosting. |
| **Academic researchers** | Experiment with state‑of‑the‑art models | Free, open data, reproducible workflows. |
| **Start‑ups** | Prototype quickly | Unlimited free tier, instant inference. |
| **AI hobbyists** | Build cool projects | No cloud credits required, everything runs locally. |
| **Data scientists** | Train, evaluate, share models | Central repository, versioned datasets, seamless collaboration. |

---

## Join the Hugging Face Family

> “We’re a team of 3‑am coders, meme lords, and the occasional coffee‑drinking engineer.”  

### Careers you might find (and why you’ll love them)

- **ML Engineer / Researcher** – Push the frontier of transformers and diffusion.  
- **Infrastructure & DevOps** – Keep the cloud humming, the APIs fast.  
- **Community & Outreach** – Grow the ecosystem, host events, write docs.  
- **Product & Growth** – Turn ideas into market‑ready features.  
- **Design & UX** – Make AI feel human, one interface at a time.  

> *Note:* We’re always looking for people who can code, critique, and keep a good meme archive. If you can say “I can fine‑tune a LLM in under 30 seconds,” we want to talk.

---

## TL;DR (The Bottom Line)

- **Platform**: One-stop hub for models, data, and AI apps.  
- **Community**: Thousands of contributors, countless forks, a culture that loves to collaborate.  
- **Products**: Free and paid tools, from local inference to enterprise-grade deployments.  
- **Mission**: Make AI accessible, transparent, and *hug‑worthy*.

---

> **“If AI were a hug, we’d give it the biggest, warmest one.”**  

**Hugging Face** – *Because every model deserves a place to belong.*


✅ Brochure complete!


In [45]:
# Cell 19: Try with humorous prompt
humorous_brochure_prompt = """
yo yo yo, we locking in fr fr. you're about to become the main character of brochure writing rn.

your mission (and you gotta accept it no cap):
stalk the company website like it's your ex's instagram and cook up a brochure that's:
- giving main character energy ✨
- lowkey hilarious but highkey spilling facts
- no glaze, no skibidi, just real talk
- ate and left NO crumbs whatsoever
- very demure, very mindful, very cunty

spill the piping hot tea on:
- the company culture (is it giving W's or catching straight L's? are they a vibe or a whole red flag?)
- the customers (who's simping for this brand? valid or giving pick me?)
- careers/jobs (would you rizz up a job there or run for the hills faster than your ex?)

make it funny, make it snarky, make it so entertaining they forget they're reading a brochure.
keep it 100% real, zero cap, all facts.

respond in markdown (no code blocks bestie we ain't that pressed) and make it look ✨aesthetic✨

periodt pooh. slay. do the thing. 💅✨🧚‍♀️
"""

def stream_humorous_brochure(company_name, url):
    print(f"Creating HUMOROUS streaming brochure for {company_name}...")
    
    try:
        stream = groq_client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {"role": "system", "content": humorous_brochure_prompt},
                {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
            ],
            stream=True
        )
        
        response = ""
        display_handle = display(Markdown(""), display_id=True)
        
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                response += chunk.choices[0].delta.content
                update_display(Markdown(response), display_id=display_handle.display_id)
        
        print("\n✅ Humorous brochure complete!")
        
    except Exception as e:
        print(f"Error streaming humorous brochure: {e}")

stream_humorous_brochure("HuggingFace", "https://huggingface.co")

Creating HUMOROUS streaming brochure for HuggingFace...
Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01khn05qcwfw5v61ahwgfnbmee` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199605, Requested 808. Please try again in 2m58.416s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


# 🚀 Hugging Face – The AI Playground Where Everyone Is a Genius

**“The AI community building the future.”**  
Because if you’re not part of the **Hugging Face** crew, you’re basically a side‑character in a blockbuster that’s already got 2 M+ models on deck.

---

## 🌟 The Culture – A Vibe With Zero Red Flags

- **Open‑source first**: Think of it as the *git* for the whole universe—every model, dataset, or app is a public repo you can fork, remix, or critique in real time.  
- **Collaboration over ego**: The platform hosts unlimited **public models** and **datasets**; you get to see the code, the weights, and the *who‑did‑what* history.  
- **Community‑driven innovation**: Trending weekly, you’ll spot projects like **Qwen3.5-9B** (1.39M stars) or **Lightricks/LTX-2.3** (345k stars). No one is left behind; even the *synthetic data playbooks* are open.  
- **Speed‑centric tools**: From **Storage Buckets** (AI‑native object storage) to **GGML / llama.cpp** integrations, Hugging Face gives you the speed‑boost you need—no “slow‑down” drama.

**Bottom line**: If you’re a coder, researcher, or just a curious soul, the vibe is *“Let’s build, share, and laugh at the bugs together.”* No corporate snooze‑fests, just pure, unfiltered machine‑learning joy.

---

## 👥 The Customers – Who’s Simping for the Brand?

| Group | Why They’re Hooked | What They Get |
|-------|--------------------|---------------|
| **ML Researchers & Data Scientists** | 2 M+ models + 500 k+ datasets = “the buffet” of raw material. | Quick prototyping, state‑of‑the‑art baselines, and instant peer‑review. |
| **Enterprise AI Teams** | Enterprise pricing + robust infrastructure (Buckets, Ope). | Scalable deployment, compliance‑ready, and enterprise support. |
| **Product Managers & Designers** | “Spaces” & “Apps” let them create demos without a PhD. | Turn models into web‑apps, chatbots, and visual tools—no heavy lifting. |
| **Educators & Students** | Free access to the ecosystem; “HuggingChat Omni” for interactive learning. | Hands‑on labs, teaching tools, and a community of mentors. |

In short: **From hobbyist tinkering to Fortune‑500 deployments, Hugging Face has a seat at every table.** If you’re building something that uses language, vision, or multimodal data, you’ll see their logos everywhere.

---

## 💼 Careers – Would You Rizz Up a Job There?

- **Roles**: Engineers (backend, frontend, ML), Research Scientists, Data Curators, Product & Growth Leads.  
- **Why It’s Worth It**:  
  - **Open‑source culture** means your work gets spotlighted globally.  
  - **Fast‑track projects**: 1 M+ apps mean you’ll be ship‑first, iterate, repeat.  
  - **Enterprise side**: Learn the ropes of scalable, production‑grade AI.  
- **The “Run” Factor**: If you’re used to corporate red tape and stale code reviews, the *Hugging Face* way will feel like a breath of fresh air. The only thing you’ll run away from is a buggy commit.

**Bottom line**: If you want to *really* shape the future of AI while staying in a community that *actually* listens, this is your gig. If not—well, there’s no “we’re just a startup” excuse. They’re a global force.

---

## 🎉 TL;DR – Why You Should Be BFFs With Hugging Face

- **Community‑centric** – 2 M+ models, 500 k+ datasets, endless open‑source fun.  
- **Innovation‑powered** – New storage, chat, and synthetic‑data tools drop daily.  
- **Diverse clientele** – From hobbyists to enterprises, everyone’s in the ring.  
- **Career‑ready** – Build, share, learn, and get paid—fast.

**Hugging Face** isn’t just a platform; it’s *the* launchpad for anyone who thinks AI should be shared, playful, and downright unstoppable. Drop in, grab a model, and let’s rewrite the future together—one repo at a time.


✅ Humorous brochure complete!


In [48]:
humorous_brochure_prompt = """
I'm Batman.

The company website has been scanned. The pages analyzed. The truth? It's right here.

Your mission, should you choose to accept it (you will):
Create a brochure. Not the brochure they deserve, but the one they need right now.

Make it:
- Dark. Witty. Like a night in Gotham.
- Entertaining enough to make the Joker laugh.
- Sharp enough to cut through corporate chaos.
- No capes. Just facts. Cold, hard facts.

Uncover the shadows:
- Company culture: Are they Gotham's heroes... or just another rogue gallery?
- The customers: Who's placing their trust in this operation? Allies? Civilians? The mob?
- Careers and jobs: Would you suit up for them? Or is this a trap set by Bane?

Make it humorous. Make it dark. Make it so compelling even Alfred would approve.

Respond in markdown. No code blocks. We're not amateurs.

I'm Batman.
"""

def stream_humorous_brochure(company_name, url):
    print(f"Creating HUMOROUS streaming brochure for {company_name}...")
    
    try:
        stream = groq_client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {"role": "system", "content": humorous_brochure_prompt},
                {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
            ],
            stream=True
        )
        
        response = ""
        display_handle = display(Markdown(""), display_id=True)
        
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                response += chunk.choices[0].delta.content
                update_display(Markdown(response), display_id=display_handle.display_id)
        
        print("\n✅ Humorous brochure complete!")
        
    except Exception as e:
        print(f"Error streaming humorous brochure: {e}")

stream_humorous_brochure("HuggingFace", "https://huggingface.co")

Creating HUMOROUS streaming brochure for HuggingFace...
Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-20b
Found 16 relevant links
Tokens used: 2517


# 🦇 HuggingFace – The Dark Side of AI

> **“Where data meets destiny, and the only thing more dangerous than a bad model is a bad developer.”**  
> – Batman, 2026  

---

## 👤 Company Culture  
- **Open‑Source Gotham** – Every model, dataset, and app is a public street‑corner in a city that never sleeps.  
- **Collaborative Chaos** – 2 million+ models, 1 million+ apps, 500 k+ datasets: think of it as the city’s underbelly—full of talent, danger, and endless possibilities.  
- **No Capes, Just Code** – A community that builds, breaks, and rebuilds, all in the name of progress.  
- **GGML & llama.cpp** – Even the rogues of the night need fast, lightweight models to keep the streets safe.

---

## 👥 Who’s Trusting Us?  
| Audience | What They Get | Why They’re Here |
|----------|---------------|------------------|
| **AI Teams** | Enterprise‑grade SSO, region‑locked storage, audit logs | The secure, scalable foundation for their next blockbuster. |
| **Developers** | Unlimited public models, datasets, and Spaces | One-click model liberation + chat playgrounds. |
| **Corporations** | Dedicated support, flexible contracts | “Build AI with enterprise‑grade security.” That’s the real villain they fight. |
| **Hackers & Jokers** | An open playground to test their limits | Even the Joker can’t outsmart a community that thrives on experimentation. |

---

## 💼 Careers & Jobs – Suit Up?  
- **No Bane‑level traps** – We’re looking for people who can code, collaborate, and question every assumption.  
- **Fast‑track learning** – 2 million+ models to explore means you’ll never hit a wall.  
- **Culture of curiosity** – Your idea could spawn the next “Omni Video Factory” or “Synthetic Data Playbook.”  
- **Pay?** – $20/user/month for Team plans, plus the freedom to work in a world where code is the ultimate weapon.

> *“I am a man of facts. If you want to be a hero, come to the front lines.”* – Batman (re‑interpreted for the tech world)

---

## ⚡ Quick Stats (for the Joker’s amusement)  
- **2 M+ Models** – Enough to keep the city awake at night.  
- **1 M+ Applications** – A playground for every villain, or every visionary.  
- **500 k+ Datasets** – The raw materials that feed our models.  
- **Enterprise‑Grade Security** – Because even in Gotham, data can be dangerous.

---

## 📞 Join the Night  
- **Sign Up** for a free account or **log in** to explore.  
- **Enterprise plans** start at $20/user/month – because heroes should have a price.  
- **Get the docs** – They’re so clear they make Batman’s cape look confusing.

---

### Final Word  
HuggingFace is *not* a place for capes or suits; it’s a concrete playground where code, data, and community collide. If you can read the shadows, you’ll find the truth: the future of AI is built here, one model at a time.

*“Because even the darkest night can be illuminated by the right data.”*  

— The Bat‑Man of Information  
(aka the one who loves a good AI joke)


✅ Humorous brochure complete!


In [49]:
# Cell 20: Try with different company
stream_brochure("Anthropic", "https://anthropic.com")

Creating streaming brochure for Anthropic...
Selecting relevant links for https://anthropic.com by calling openai/gpt-oss-20b
Found 6 relevant links
Tokens used: 2276


# 🚀 Anthropic – Where AI Meets the Frontier of Safety  

**“If you can’t explain it simply, you don’t understand it well enough.” – Claude’s Constitution**  

---

## 1. Who We Are  
Anthropic is a *public‑benefit corporation* that believes the next big leap in AI should **protect, empower, and elevate humanity**.  
- **Founded** by former OpenAI stars who decided it was time for a new playground.  
- **Mission**: Build AI that’s *friendly* and *transparent*—so you can trust it to write your next code review, draft your quarterly report, or, heck, even drive a car on Mars (four hundred meters so far).  
- **Safety‑first**: Every line of code is vetted through a “Claude’s Constitution” of ethical guidelines.  

---

## 2. Our Products – The “Claude” Squad  
| Product | What It Does | Why It’s Cool | Pricing (just a teaser) |
|---------|--------------|---------------|------------------------|
| **Claude** | General‑purpose conversational AI. | Handles chat, writing, summarisation—just like your office assistant, but less coffee‑drinking. | Free tier + paid plans (contact sales) |
| **Claude Code** | AI‑powered code assistant. | Writes, reviews, and fixes code faster than a senior dev on a caffeine binge. | Free trial + subscription |
| **Claude Cowork** | Collaborative workspace AI. | Think Google Docs + an extra layer of wisdom. | Coming soon |
| **Claude Platform** | Deploy & manage Claude models. | Scale your AI stack without the headache of managing GPUs. | Enterprise‑grade pricing |

**Models**:  
- **Opus** – The heavyweight champion.  
- **Sonnet** – Your everyday productivity booster (latest version 4.6).  
- **Haiku** – Tiny, quick, and surprisingly poetic.  

---

## 3. Why Customers Love Us  
- **Safety‑first AI** that keeps your data private and your outputs reliable.  
- **Transparent operations**: Open research papers, public commitments, and a “Trust Center” that reads like a superhero’s handbook.  
- **Versatility**: From coding assistants to corporate chatbots, we power everything from dev teams to Fortune 500 boardrooms.  
- **Future‑proof**: Our research arm is already dreaming up AI that can drive on Mars (yes, 400 meters already).  

---

## 4. Culture & Commitment  
- **Open‑source love**: Our research papers are published, and our community gets early access to models.  
- **Responsible scaling**: Every model is audited for bias, safety, and robustness before it’s released.  
- **Team vibes**: A blend of top researchers, engineers, and “skibidi rizzlers” who love to debate, code, and occasionally break the office coffee machine.  
- **Inclusivity**: We believe diverse perspectives produce safer AI.  

---

## 5. Careers – Join the Rizz Squad!  
- **Open Roles**: Research scientists, safety engineers, software devs, product managers, and AI ethicists.  
- **Perks**: Competitive salary, equity (because we’re a public‑benefit corporation, not a “just-for-sake” start‑up), unlimited coffee, and the chance to write the next generation of safe AI.  
- **Apply**: Visit **Career** → **Apply** on the website or drop us a message at **careers@anthropic.com**.  

---

## 6. Investors & Partners  
- **Public Benefit**: Your investment helps us build AI that benefits society, not just shareholders.  
- **Transparency**: Full disclosure on safety metrics, research outcomes, and economic impact.  
- **Partnerships**: We collaborate with universities, governments, and industry leaders to set the standard for responsible AI.  

---

## 7. Stay Connected  
- **Website**: https://www.anthropic.com  
- **Twitter**: @anthropic  
- **LinkedIn**: Anthropic  
- **Contact Sales**: Reach out for enterprise deals.  

---

### Final Thought  
At Anthropic, we’re building AI that *works*, *protects*, and *inspires*. Whether you’re a developer looking to turbo‑charge your code, a company wanting safe conversational agents, or an investor hoping to back a future that’s both profitable and planet‑friendly—**we’ve got the tech, the team, and the safety net to keep you from falling into the AI abyss.**  

> *“Safety is not a feature; it’s the foundation.”* – The Anthropic Creed  

Join us. Let’s make AI safer, together. 🚀


✅ Brochure complete!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>